In [1]:
!pip install sqlalchemy psycopg2-binary

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from dotenv import load_dotenv, find_dotenv
import os


load_dotenv(find_dotenv())

DB_URL = os.getenv("LOCAL_DATABASE_URL")


if DB_URL and DB_URL.startswith("postgresql://"):
    DB_URL = DB_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(DB_URL)

print(f"Engine dibikin, target: {DB_URL.split('@')[1] if DB_URL else 'NONE'}")

Engine dibikin, target: localhost:5432/retail_analytics


In [3]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_schema();"))
    print(result.fetchone())

('retail_analytics', 'public')


In [6]:
from sqlalchemy.dialects.postgresql import JSONB
from sqlalchemy.types import TIMESTAMP

df = pd.read_json("../data/raw/operational/customers.json")

# Convert tiap baris jadi dict Python (bukan string)
raw_data = df.to_dict(orient="records")

df_bronze = pd.DataFrame({
    "raw": raw_data,
    "timestamp": datetime.utcnow()
})

df_bronze.to_sql(
    "customers",
    engine,
    schema="bronze",
    if_exists="replace",
    index=False,
    dtype={
        "raw": JSONB,
        "timestamp": TIMESTAMP(timezone=True)
    }
)

print("Selesai, cek pgAdmin lagi")

Selesai, cek pgAdmin lagi


In [7]:
from pathlib import Path

RAW_DIR = Path("../data/raw")

def load_json_to_bronze(file_path: str, table_name: str):
    """Load 1 file JSON ke bronze schema dengan struktur raw JSONB + timestamp."""
    full_path = RAW_DIR / file_path
    df = pd.read_json(full_path)
    
    raw_data = df.to_dict(orient="records")
    df_bronze = pd.DataFrame({
        "raw": raw_data,
        "timestamp": datetime.utcnow()
    })
    
    df_bronze.to_sql(
        table_name,
        engine,
        schema="bronze",
        if_exists="replace",
        index=False,
        dtype={
            "raw": JSONB,
            "timestamp": TIMESTAMP(timezone=True)
        }
    )
    
    print(f"OK {table_name}: {len(df_bronze)} baris")

# Test dengan file lain: orders.json
load_json_to_bronze("operational/orders.json", "orders")

OK orders: 10001 baris


In [8]:
# Mapping: nama tabel to path file (relatif ke data/raw/)
JSON_FILES = {
    # operational (11 file)
    "customers":            "operational/customers.json",
    "customer_profiles":    "operational/customer_profiles.json",
    "customer_addresses":   "operational/customer_addresses.json",
    "products":             "operational/products.json",
    "product_categories":   "operational/product_categories.json",
    "stores":               "operational/stores.json",
    "sales_channels":       "operational/sales_channels.json",
    "promotions":           "operational/promotions.json",
    "orders":               "operational/orders.json",
    "order_items":          "operational/order_items.json",
    "order_promotions":     "operational/order_promotions.json",
    # events (5 file)
    "payment_events":       "events/payment_events.json",
    "refund_events":        "events/refund_events.json",
    "return_events":        "events/return_events.json",
    "support_events":       "events/support_events.json",
    "web_events":           "events/web_events.json",
    # reference (1 file JSON, sisanya CSV nanti)
    "city_reference":       "reference/city_reference.json",
}

for table_name, file_path in JSON_FILES.items():
    load_json_to_bronze(file_path, table_name)

print(f"\nSelesai! Total {len(JSON_FILES)} tabel JSON di bronze.")

OK customers: 2500 baris
OK customer_profiles: 2777 baris
OK customer_addresses: 2500 baris
OK products: 80 baris
OK product_categories: 86 baris
OK stores: 5 baris
OK sales_channels: 4 baris
OK promotions: 12 baris
OK orders: 10001 baris
OK order_items: 24960 baris
OK order_promotions: 15729 baris
OK payment_events: 22582 baris
OK refund_events: 4121 baris
OK return_events: 3553 baris
OK support_events: 2971 baris
OK web_events: 30986 baris
OK city_reference: 8 baris

Selesai! Total 17 tabel JSON di bronze.


In [9]:
def load_csv_to_bronze(file_path: str, table_name: str):
    """Load 1 file CSV ke bronze schema. Kolom asli dipertahankan + tambah timestamp."""
    full_path = RAW_DIR / file_path
    df = pd.read_csv(full_path)
    
    # Tambah kolom timestamp sebagai lineage
    df["timestamp"] = datetime.utcnow()
    
    df.to_sql(
        table_name,
        engine,
        schema="bronze",
        if_exists="replace",
        index=False,
        dtype={"timestamp": TIMESTAMP(timezone=True)}
    )
    
    print(f"OK {table_name}: {len(df)} baris, {len(df.columns)} kolom")


CSV_FILES = {
    "inventory_snapshots":  "inventory/inventory_snapshots.csv",
    "campaign_spend":       "reference/campaign_spend.csv",
}

for table_name, file_path in CSV_FILES.items():
    load_csv_to_bronze(file_path, table_name)

print(f"\nSelesai! Total tabel bronze: {17 + len(CSV_FILES)} = 19")

OK inventory_snapshots: 35040 baris, 7 kolom
OK campaign_spend: 855 baris, 5 kolom

Selesai! Total tabel bronze: 19 = 19


In [14]:
!pip install python-dotenv

  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)
